# Notebook 21 — Foundation Script-Reliance Audit

## Research Question

Do strong foundation-based writer-verification representations depend on script-sensitive information when making same-writer versus different-writer decisions?

More specifically:

- How much script information is linearly accessible from frozen and writer-adapted DINOv2 representations?
- Does high script accessibility imply high decision-level script reliance?
- How does script reliance change across DINOv2 model scales and adaptation strategies?
- Does the strong PCA-225 + writer-LDA verifier gain writer discrimination while increasing, decreasing, or preserving script-sensitive decision behavior?

## Models

- DINOv2-S
- DINOv2-B
- DINOv2-L
- DINOv2-G

## Representation Stages

For each available model, the audit will consider:

1. Frozen representation.
2. Direct writer-LDA representation.
3. PCA-225 followed by writer-LDA.

The PCA dimensionality is fixed at 225 from Notebook 20 and will not be tuned further on validation data.

## Audit Dimensions

The analysis will separate three concepts:

1. **Writer-verification performance**
   - overall ROC-AUC
   - EER
   - cross-script macro AUC

2. **Script accessibility**
   - how accurately an independent linear probe can decode script from the representation

3. **Decision-level script reliance**
   - how much verification similarity scores change after removing script-sensitive representation directions
   - comparison against dimension-matched random-subspace interventions

## Experimental Discipline

- Script probes and script-sensitive directions are learned only from the development split.
- Writer verification and intervention sensitivity are evaluated only on the fixed validation protocol.
- The official test split remains untouched.
- No PCA dimensionality search or validation-driven representation tuning is allowed.
- Script-subspace removal is interpreted as a representation intervention and decision-sensitivity diagnostic, not as proof of a causal effect.
- High script accessibility is not assumed to imply high decision reliance.
- Any proposed novelty around leakage-versus-reliance remains a candidate contribution until it survives dedicated literature verification and external validation.

## Decision Rule for the Next Research Stage

- If strong foundation verifiers show substantial script-specific decision reliance, the next method stage will investigate decision-stable writer adaptation.
- If strong foundation verifiers already show low script reliance, the project will prioritize error-predictive uncertainty and selective verification instead of forcing unnecessary invariance.

In [7]:
from pathlib import Path

import json
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    path
    for path in [
        current_path,
        *current_path.parents,
    ]
    if (
        path
        / "pyproject.toml"
    ).exists()
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

VALIDATION_PAIR_PATH = (
    PROJECT_ROOT
    / "splits"
    / "verification"
    / "quwi_validation_verification_pairs.csv"
)

NOTEBOOK18_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "script_reliance_intervention"
)

NOTEBOOK19_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "modern_baseline_benchmarking"
)

NOTEBOOK20_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "dinov2_efficiency_frontier"
)

split_df = pd.read_csv(
    SPLIT_PATH
)

validation_pairs_df = pd.read_csv(
    VALIDATION_PAIR_PATH
)

development_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "development_train"
    ]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "validation"
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print()

print(
    "Development:",
    development_df[
        "writer"
    ].nunique(),
    "writers /",
    len(
        development_df
    ),
    "images",
)

print(
    "Validation:",
    validation_df[
        "writer"
    ].nunique(),
    "writers /",
    len(
        validation_df
    ),
    "images",
)

print(
    "Validation pairs:",
    len(
        validation_pairs_df
    ),
)

print()

print(
    "Notebook 18 artifacts:"
)

for path in sorted(
    NOTEBOOK18_REPORT_DIR.glob(
        "*"
    )
):
    print(
        path.name
    )

print()

print(
    "Notebook 19 DINO artifacts:"
)

for path in sorted(
    NOTEBOOK19_REPORT_DIR.glob(
        "dinov2*"
    )
):
    print(
        path.name
    )

print()

print(
    "Notebook 20 artifacts:"
)

for path in sorted(
    NOTEBOOK20_REPORT_DIR.glob(
        "*"
    )
):
    print(
        path.name
    )

print()

print(
    "Official test used:",
    False,
)

Project root: C:\Users\com\Documents\handwriting-cross-script-research

Development: 226 writers / 904 images
Validation: 56 writers / 224 images
Validation pairs: 18816

Notebook 18 artifacts:
decision_side_uncertainty_seed123.csv
decision_side_uncertainty_seed2026.csv
family_model_comparison.csv
model_reliance_summary.csv
notebook18_experiment_summary.json
paired_cross_script_reliance.csv
random_subspace_control.csv
script_subspace_removal_trajectory.csv
script_subspace_specificity.csv
strong_uncertainty_baseline_seed123.csv
strong_uncertainty_baseline_seed2026.csv

Notebook 19 DINO artifacts:
dinov2_vitg14_reg_condition_results.csv
dinov2_vitg14_reg_embeddings.npz
dinov2_vitg14_reg_summary.csv
dinov2_vitl14_reg_condition_results.csv
dinov2_vitl14_reg_embeddings.npz
dinov2_vitl14_reg_summary.csv

Notebook 20 artifacts:
accuracy_parameter_frontier.png
accuracy_parameter_landscape.png
adaptation_strategy_comparison.png
dinov2_vitb14_reg_embeddings.npz
dinov2_vits14_reg_embeddings.npz
d

In [3]:
NOTEBOOK18_SUMMARY_PATH = (
    NOTEBOOK18_REPORT_DIR
    / "notebook18_experiment_summary.json"
)

with open(
    NOTEBOOK18_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook18_summary = json.load(
        file
    )

print(
    json.dumps(
        notebook18_summary,
        indent=2,
    )
)

{
  "experiment": "Script-Reliance Intervention for Cross-Script Writer Verification",
  "selected_script_subspace_dimension": 128,
  "selection_protocol": "Development-only writer-disjoint subspace-train and monitor split",
  "batch_alt_seed123": {
    "original_writer_auc": 0.7764246418264277,
    "intervened_writer_auc": 0.7765837585034013,
    "residual_script_auc": 0.8254145408163265,
    "mean_script_reliance": 0.009797601673933388
  },
  "batch_alt_seed2026": {
    "original_writer_auc": 0.7700305832560297,
    "intervened_writer_auc": 0.7706360479797979,
    "residual_script_auc": 0.7010522959183674,
    "mean_script_reliance": 0.010207820265921593
  },
  "cross_script_reliance": {
    "mean_relative_control_to_batch_reduction": 0.7264590822660123,
    "control_mean_intervention_auc_change": 0.011843945674860912,
    "batch_alt_mean_intervention_auc_change": -0.0011076465387292456
  },
  "random_subspace_control": {
    "seed123_control_script_to_random_reliance_ratio": 3.44983

In [4]:
notebook18_artifact_names = [
    "model_reliance_summary.csv",
    "script_subspace_removal_trajectory.csv",
    "script_subspace_specificity.csv",
    "random_subspace_control.csv",
    "paired_cross_script_reliance.csv",
]

for filename in notebook18_artifact_names:
    path = (
        NOTEBOOK18_REPORT_DIR
        / filename
    )

    artifact_df = pd.read_csv(
        path
    )

    print(
        "=" * 80
    )

    print(
        filename
    )

    print(
        "Shape:",
        artifact_df.shape
    )

    print(
        "Columns:",
        artifact_df.columns.tolist()
    )

    print()

    print(
        artifact_df
        .head(8)
        .to_string(
            index=False
        )
    )

    print()

model_reliance_summary.csv
Shape: (4, 11)
Columns: ['training_seed', 'model', 'original_writer_auc', 'intervened_writer_auc', 'residual_script_auc', 'mean_script_reliance', 'reliance_error_auc', 'combined_auc_gain', 'combined_auc_better', 'aurc_better', 'intervention_auc_change']

 training_seed               model  original_writer_auc  intervened_writer_auc  residual_script_auc  mean_script_reliance  reliance_error_auc  combined_auc_gain  combined_auc_better  aurc_better  intervention_auc_change
           123     lambda0_control             0.738789               0.768275             0.891103              0.063327            0.617219           0.020262                   35           37                 0.029486
           123 batch_alt_lambda0p5             0.776425               0.776584             0.825415              0.009798            0.592974           0.022980                   40           40                 0.000159
          2026     lambda0_control             0.738279   

In [5]:
NOTEBOOK18_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "18_script_reliance_intervention.ipynb"
)

with open(
    NOTEBOOK18_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook18_source = json.load(
        file
    )

subspace_keywords = [
    "selected_script_subspace_dimension",
    "removed_dimensions",
    "monitor_script_auc",
    "script_subspace",
    "subspace_train",
    "subspace_monitor",
    "logisticregression",
]

for cell_index, cell in enumerate(
    notebook18_source[
        "cells"
    ]
):
    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    if any(
        keyword in source.lower()
        for keyword in subspace_keywords
    ):
        print(
            "=" * 100
        )

        print(
            "Notebook 18 code cell:",
            cell_index
        )

        print()

        print(
            source
        )

        print()

Notebook 18 code cell: 1

import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from torch.utils.data import DataLoader
from torchvision.models import resnet18

from handwriting_cross_script_research.dataset import QUWIDataset

Notebook 18 code cell: 5

script_probe = Pipeline(
    [
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=PROBE_SEED,
            ),
        ),
    ]
)

script_probe.fit(
    development_embeddings,
    development_script_labels,
)

validation_script_probabilities = (
    script_probe.predict_pr

In [6]:
intervention_keywords = [
    "random_cross_reliance",
    "script_cross_reliance",
    "mean_script_reliance",
    "random_subspace",
    "intervened_writer_auc",
    "script_reliance",
    "project_out",
]

for cell_index, cell in enumerate(
    notebook18_source[
        "cells"
    ]
):
    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    if any(
        keyword in source.lower()
        for keyword in intervention_keywords
    ):
        print(
            "=" * 100
        )

        print(
            "Notebook 18 code cell:",
            cell_index
        )

        print()

        print(
            source
        )

        print()

Notebook 18 code cell: 2

SPLIT_SEED = 42
PROBE_SEED = 42
EMBEDDING_BATCH_SIZE = 16

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate project root."
    )

IMAGE_DIR = (
    Path.home()
    / "Documents"
    / "Handwriting"
    / "QUWI"
    / "extracted"
    / "images"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

VALIDATION_PAIR_PATH = (
    PROJECT_ROOT
    / "splits"
    / "verification"
    / "quwi_validation_verification_pairs.csv"
)

NOTEBOOK16_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "script_adversarial_metric_learning"
)

NOTEBOOK17_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "multiseed_adversarial_robustness"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "script_reliance_intervention"
)

REPORT_DIR.mk

In [8]:
MODEL_ARTIFACT_PATHS = {
    "dinov2_s": (
        NOTEBOOK20_REPORT_DIR
        / "dinov2_vits14_reg_embeddings.npz"
    ),
    "dinov2_b": (
        NOTEBOOK20_REPORT_DIR
        / "dinov2_vitb14_reg_embeddings.npz"
    ),
    "dinov2_l": (
        NOTEBOOK19_REPORT_DIR
        / "dinov2_vitl14_reg_embeddings.npz"
    ),
    "dinov2_g": (
        NOTEBOOK19_REPORT_DIR
        / "dinov2_vitg14_reg_embeddings.npz"
    ),
}

development_writer_map = dict(
    zip(
        development_df["filename"],
        development_df["writer"],
    )
)

validation_writer_map = dict(
    zip(
        validation_df["filename"],
        validation_df["writer"],
    )
)

script_values = sorted(
    split_df[
        "language"
    ]
    .astype(str)
    .unique()
    .tolist()
)

assert len(
    script_values
) == 2

SCRIPT_TO_LABEL = {
    script: index
    for index, script in enumerate(
        script_values
    )
}

development_script_map = {
    filename: SCRIPT_TO_LABEL[
        str(language)
    ]
    for filename, language in zip(
        development_df["filename"],
        development_df["language"],
    )
}

validation_script_map = {
    filename: SCRIPT_TO_LABEL[
        str(language)
    ]
    for filename, language in zip(
        validation_df["filename"],
        validation_df["language"],
    )
}

raw_artifacts = {}

for model_name, artifact_path in (
    MODEL_ARTIFACT_PATHS.items()
):
    with np.load(
        artifact_path
    ) as artifact:
        raw_artifacts[
            model_name
        ] = {
            key: artifact[
                key
            ].copy()
            for key in artifact.files
        }

    development_filenames = (
        raw_artifacts[
            model_name
        ][
            "development_filenames"
        ]
        .astype(str)
    )

    validation_filenames = (
        raw_artifacts[
            model_name
        ][
            "validation_filenames"
        ]
        .astype(str)
    )

    development_writers = np.array(
        [
            development_writer_map[
                filename
            ]
            for filename in development_filenames
        ],
        dtype=np.int64,
    )

    validation_writers = np.array(
        [
            validation_writer_map[
                filename
            ]
            for filename in validation_filenames
        ],
        dtype=np.int64,
    )

    development_scripts = np.array(
        [
            development_script_map[
                filename
            ]
            for filename in development_filenames
        ],
        dtype=np.int64,
    )

    validation_scripts = np.array(
        [
            validation_script_map[
                filename
            ]
            for filename in validation_filenames
        ],
        dtype=np.int64,
    )

    if (
        "development_writers"
        in raw_artifacts[
            model_name
        ]
    ):
        assert np.array_equal(
            development_writers,
            raw_artifacts[
                model_name
            ][
                "development_writers"
            ].astype(
                np.int64
            ),
        )

    raw_artifacts[
        model_name
    ][
        "development_filenames"
    ] = development_filenames

    raw_artifacts[
        model_name
    ][
        "validation_filenames"
    ] = validation_filenames

    raw_artifacts[
        model_name
    ][
        "development_writers_aligned"
    ] = development_writers

    raw_artifacts[
        model_name
    ][
        "validation_writers_aligned"
    ] = validation_writers

    raw_artifacts[
        model_name
    ][
        "development_scripts"
    ] = development_scripts

    raw_artifacts[
        model_name
    ][
        "validation_scripts"
    ] = validation_scripts


print(
    "Script mapping:",
    SCRIPT_TO_LABEL,
)

print()

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    artifact = raw_artifacts[
        model_name
    ]

    print(
        model_name,
        "| development:",
        artifact[
            "development_embeddings"
        ].shape,
        "| validation:",
        artifact[
            "validation_embeddings"
        ].shape,
        "| writers:",
        len(
            np.unique(
                artifact[
                    "development_writers_aligned"
                ]
            )
        ),
        "| development script counts:",
        np.bincount(
            artifact[
                "development_scripts"
            ]
        ).tolist(),
        "| validation script counts:",
        np.bincount(
            artifact[
                "validation_scripts"
            ]
        ).tolist(),
    )

Script mapping: {'Arabic': 0, 'English': 1}

dinov2_s | development: (904, 384) | validation: (224, 384) | writers: 226 | development script counts: [452, 452] | validation script counts: [112, 112]
dinov2_b | development: (904, 768) | validation: (224, 768) | writers: 226 | development script counts: [452, 452] | validation script counts: [112, 112]
dinov2_l | development: (904, 1024) | validation: (224, 1024) | writers: 226 | development script counts: [452, 452] | validation script counts: [112, 112]
dinov2_g | development: (904, 1536) | validation: (224, 1536) | writers: 226 | development script counts: [452, 452] | validation script counts: [112, 112]


In [9]:
COMMON_PCA_DIM = 225
FIXED_SCRIPT_SUBSPACE_RANK = 56
SCRIPT_TRAJECTORY_RANKS = [
    1,
    2,
    4,
    8,
    16,
    32,
    56,
]


def normalize_rows(
    embeddings,
):
    norms = np.linalg.norm(
        embeddings,
        axis=1,
        keepdims=True,
    )

    if np.any(
        norms < 1e-12
    ):
        raise RuntimeError(
            "Zero-norm embedding encountered."
        )

    return (
        embeddings
        / norms
    )


def build_representation_stages(
    raw_development_embeddings,
    raw_validation_embeddings,
    development_writers,
):
    raw_development_embeddings = (
        normalize_rows(
            raw_development_embeddings
        )
    )

    raw_validation_embeddings = (
        normalize_rows(
            raw_validation_embeddings
        )
    )

    direct_lda = (
        LinearDiscriminantAnalysis(
            solver="svd"
        )
    )

    direct_lda.fit(
        raw_development_embeddings,
        development_writers,
    )

    direct_development = normalize_rows(
        direct_lda.transform(
            raw_development_embeddings
        )
    )

    direct_validation = normalize_rows(
        direct_lda.transform(
            raw_validation_embeddings
        )
    )

    pca = PCA(
        n_components=COMMON_PCA_DIM,
        svd_solver="full",
    )

    development_pca = pca.fit_transform(
        raw_development_embeddings
    )

    validation_pca = pca.transform(
        raw_validation_embeddings
    )

    pca_writer_lda = (
        LinearDiscriminantAnalysis(
            solver="svd"
        )
    )

    pca_writer_lda.fit(
        development_pca,
        development_writers,
    )

    pca_lda_development = normalize_rows(
        pca_writer_lda.transform(
            development_pca
        )
    )

    pca_lda_validation = normalize_rows(
        pca_writer_lda.transform(
            validation_pca
        )
    )

    return {
        "frozen": {
            "development": (
                raw_development_embeddings
            ),
            "validation": (
                raw_validation_embeddings
            ),
        },
        "direct_lda": {
            "development": (
                direct_development
            ),
            "validation": (
                direct_validation
            ),
        },
        "pca225_lda": {
            "development": (
                pca_lda_development
            ),
            "validation": (
                pca_lda_validation
            ),
        },
        "pca_variance_retained": float(
            pca.explained_variance_ratio_.sum()
        ),
    }


representation_registry = {}
representation_inventory_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    artifact = raw_artifacts[
        model_name
    ]

    stages = build_representation_stages(
        artifact[
            "development_embeddings"
        ],
        artifact[
            "validation_embeddings"
        ],
        artifact[
            "development_writers_aligned"
        ],
    )

    for stage_name in [
        "frozen",
        "direct_lda",
        "pca225_lda",
    ]:
        stage = stages[
            stage_name
        ]

        representation_registry[
            (
                model_name,
                stage_name,
            )
        ] = {
            "development_embeddings": stage[
                "development"
            ],
            "validation_embeddings": stage[
                "validation"
            ],
            "development_filenames": artifact[
                "development_filenames"
            ],
            "validation_filenames": artifact[
                "validation_filenames"
            ],
            "development_writers": artifact[
                "development_writers_aligned"
            ],
            "validation_writers": artifact[
                "validation_writers_aligned"
            ],
            "development_scripts": artifact[
                "development_scripts"
            ],
            "validation_scripts": artifact[
                "validation_scripts"
            ],
        }

        representation_inventory_rows.append(
            {
                "model": model_name,
                "stage": stage_name,
                "dimension": int(
                    stage[
                        "development"
                    ].shape[
                        1
                    ]
                ),
                "development_mean_norm": float(
                    np.linalg.norm(
                        stage[
                            "development"
                        ],
                        axis=1,
                    ).mean()
                ),
                "validation_mean_norm": float(
                    np.linalg.norm(
                        stage[
                            "validation"
                        ],
                        axis=1,
                    ).mean()
                ),
                "pca_variance_retained": (
                    stages[
                        "pca_variance_retained"
                    ]
                    if stage_name
                    == "pca225_lda"
                    else np.nan
                ),
            }
        )


representation_inventory_df = (
    pd.DataFrame(
        representation_inventory_rows
    )
)

print(
    "Fixed script-sensitive subspace rank:",
    FIXED_SCRIPT_SUBSPACE_RANK,
)

print(
    "Trajectory ranks:",
    SCRIPT_TRAJECTORY_RANKS,
)

print()

print(
    representation_inventory_df
    .round(6)
    .to_string(
        index=False
    )
)

Fixed script-sensitive subspace rank: 56
Trajectory ranks: [1, 2, 4, 8, 16, 32, 56]

   model      stage  dimension  development_mean_norm  validation_mean_norm  pca_variance_retained
dinov2_s     frozen        384                    1.0                   1.0                    NaN
dinov2_s direct_lda        225                    1.0                   1.0                    NaN
dinov2_s pca225_lda        225                    1.0                   1.0               0.995223
dinov2_b     frozen        768                    1.0                   1.0                    NaN
dinov2_b direct_lda        225                    1.0                   1.0                    NaN
dinov2_b pca225_lda        225                    1.0                   1.0               0.987208
dinov2_l     frozen       1024                    1.0                   1.0                    NaN
dinov2_l direct_lda        225                    1.0                   1.0                    NaN
dinov2_l pca225_lda     

In [10]:
g_development_embeddings = normalize_rows(
    raw_artifacts[
        "dinov2_g"
    ][
        "development_embeddings"
    ]
)

g_development_writers = raw_artifacts[
    "dinov2_g"
][
    "development_writers_aligned"
]

g_pca_check = PCA(
    n_components=COMMON_PCA_DIM,
    svd_solver="full",
)

g_development_pca = g_pca_check.fit_transform(
    g_development_embeddings
)

g_lda_check = LinearDiscriminantAnalysis(
    solver="svd"
)

g_lda_check.fit(
    g_development_pca,
    g_development_writers,
)

g_development_lda = g_lda_check.transform(
    g_development_pca
)

adapted_dimensions = [
    value[
        "development_embeddings"
    ].shape[
        1
    ]
    for key, value in representation_registry.items()
    if key[
        1
    ] in [
        "direct_lda",
        "pca225_lda",
    ]
]

print(
    "Development writers:",
    len(
        np.unique(
            g_development_writers
        )
    ),
)

print(
    "Theoretical maximum LDA dimension:",
    len(
        np.unique(
            g_development_writers
        )
    )
    - 1,
)

print(
    "PCA output shape:",
    g_development_pca.shape,
)

print(
    "PCA matrix rank:",
    np.linalg.matrix_rank(
        g_development_pca
    ),
)

print(
    "LDA transformed shape:",
    g_development_lda.shape,
)

print(
    "LDA scalings shape:",
    g_lda_check.scalings_.shape,
)

print(
    "LDA explained-variance components:",
    len(
        g_lda_check.explained_variance_ratio_
    ),
)

print(
    "Registry G PCA-LDA dimension:",
    representation_registry[
        (
            "dinov2_g",
            "pca225_lda",
        )
    ][
        "development_embeddings"
    ].shape[
        1
    ],
)

print(
    "Minimum adapted representation dimension:",
    min(
        adapted_dimensions
    ),
)

print(
    "Fixed intervention rank:",
    FIXED_SCRIPT_SUBSPACE_RANK,
)

print(
    "Fraction of minimum adapted dimension:",
    FIXED_SCRIPT_SUBSPACE_RANK
    / min(
        adapted_dimensions
    ),
)

Development writers: 226
Theoretical maximum LDA dimension: 225
PCA output shape: (904, 225)
PCA matrix rank: 225
LDA transformed shape: (904, 224)
LDA scalings shape: (225, 224)
LDA explained-variance components: 225
Registry G PCA-LDA dimension: 224
Minimum adapted representation dimension: 224
Fixed intervention rank: 56
Fraction of minimum adapted dimension: 0.25


In [11]:
CONDITION_BY_PAGE_PAIR = {
    (1, 2): "arabic_variable_same",
    (3, 4): "english_variable_same",
    (1, 3): "cross_variable_variable",
    (1, 4): "cross_variable_same",
    (2, 3): "cross_same_variable",
    (2, 4): "cross_same_same",
}

WITHIN_SCRIPT_CONDITIONS = [
    "arabic_variable_same",
    "english_variable_same",
]

CROSS_SCRIPT_CONDITIONS = [
    "cross_variable_variable",
    "cross_variable_same",
    "cross_same_variable",
    "cross_same_same",
]


def filename_page_id(
    filename,
):
    return int(
        Path(
            filename
        ).stem.split(
            "_"
        )[-1]
    )


def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr
    difference = fpr - fnr

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        ) != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = np.argmin(
            np.abs(
                difference
            )
        )

        return float(
            (
                fpr[
                    nearest_index
                ]
                + fnr[
                    nearest_index
                ]
            )
            / 2.0
        )

    index = crossing_indices[
        0
    ]

    x0 = difference[
        index
    ]

    x1 = difference[
        index
        + 1
    ]

    weight = (
        -x0
        / (
            x1
            - x0
        )
    )

    eer = (
        fpr[
            index
        ]
        + weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    return float(
        eer
    )


def evaluate_writer_verification(
    embeddings,
    filenames,
):
    embedding_map = dict(
        zip(
            filenames,
            embeddings,
        )
    )

    embeddings_a = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in validation_pairs_df[
                "filename_a"
            ]
        ]
    )

    embeddings_b = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in validation_pairs_df[
                "filename_b"
            ]
        ]
    )

    labels = validation_pairs_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = np.sum(
        embeddings_a
        * embeddings_b,
        axis=1,
    )

    overall_auc = roc_auc_score(
        labels,
        scores,
    )

    eer = calculate_interpolated_eer(
        labels,
        scores,
    )

    conditions = [
        CONDITION_BY_PAGE_PAIR[
            (
                filename_page_id(
                    filename_a
                ),
                filename_page_id(
                    filename_b
                ),
            )
        ]
        for filename_a, filename_b in zip(
            validation_pairs_df[
                "filename_a"
            ],
            validation_pairs_df[
                "filename_b"
            ],
        )
    ]

    condition_df = validation_pairs_df[
        [
            "pair_label"
        ]
    ].copy()

    condition_df[
        "score"
    ] = scores

    condition_df[
        "condition"
    ] = conditions

    condition_auc_rows = []

    for condition, group_df in condition_df.groupby(
        "condition"
    ):
        condition_auc_rows.append(
            {
                "condition": condition,
                "auc": roc_auc_score(
                    group_df[
                        "pair_label"
                    ],
                    group_df[
                        "score"
                    ],
                ),
            }
        )

    condition_auc_df = pd.DataFrame(
        condition_auc_rows
    )

    cross_macro_auc = condition_auc_df[
        condition_auc_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()

    within_macro_auc = condition_auc_df[
        condition_auc_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()

    return {
        "overall_auc": float(
            overall_auc
        ),
        "eer": float(
            eer
        ),
        "cross_macro_auc": float(
            cross_macro_auc
        ),
        "within_macro_auc": float(
            within_macro_auc
        ),
    }


def evaluate_script_accessibility(
    development_embeddings,
    development_scripts,
    validation_embeddings,
    validation_scripts,
):
    probe = Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42,
                ),
            ),
        ]
    )

    probe.fit(
        development_embeddings,
        development_scripts,
    )

    probabilities = probe.predict_proba(
        validation_embeddings
    )[
        :,
        1
    ]

    predictions = probe.predict(
        validation_embeddings
    )

    return {
        "script_auc": float(
            roc_auc_score(
                validation_scripts,
                probabilities,
            )
        ),
        "script_accuracy": float(
            accuracy_score(
                validation_scripts,
                predictions,
            )
        ),
    }


baseline_audit_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    for stage_name in [
        "frozen",
        "direct_lda",
        "pca225_lda",
    ]:
        representation = representation_registry[
            (
                model_name,
                stage_name,
            )
        ]

        writer_result = evaluate_writer_verification(
            representation[
                "validation_embeddings"
            ],
            representation[
                "validation_filenames"
            ],
        )

        script_result = evaluate_script_accessibility(
            representation[
                "development_embeddings"
            ],
            representation[
                "development_scripts"
            ],
            representation[
                "validation_embeddings"
            ],
            representation[
                "validation_scripts"
            ],
        )

        baseline_audit_rows.append(
            {
                "model": model_name,
                "stage": stage_name,
                "dimension": representation[
                    "validation_embeddings"
                ].shape[
                    1
                ],
                "writer_auc": writer_result[
                    "overall_auc"
                ],
                "writer_eer": writer_result[
                    "eer"
                ],
                "cross_script_auc": writer_result[
                    "cross_macro_auc"
                ],
                "within_script_auc": writer_result[
                    "within_macro_auc"
                ],
                "script_probe_auc": script_result[
                    "script_auc"
                ],
                "script_probe_accuracy": script_result[
                    "script_accuracy"
                ],
            }
        )


baseline_audit_df = pd.DataFrame(
    baseline_audit_rows
)

print(
    baseline_audit_df
    .round(6)
    .to_string(
        index=False
    )
)

   model      stage  dimension  writer_auc  writer_eer  cross_script_auc  within_script_auc  script_probe_auc  script_probe_accuracy
dinov2_s     frozen        384    0.701296    0.359957          0.686559           0.771765          1.000000               1.000000
dinov2_s direct_lda        225    0.846922    0.221916          0.804132           0.930954          0.638473               0.616071
dinov2_s pca225_lda        225    0.899126    0.173864          0.868025           0.962593          1.000000               1.000000
dinov2_b     frozen        768    0.642238    0.410714          0.618377           0.763552          1.000000               1.000000
dinov2_b direct_lda        225    0.737003    0.324405          0.696308           0.817732          0.549107               0.558036
dinov2_b pca225_lda        225    0.914989    0.160714          0.890048           0.969614          1.000000               1.000000
dinov2_l     frozen       1024    0.579876    0.455357          0.586

In [12]:
PROBE_SEED = 42
FIXED_SCRIPT_SUBSPACE_RANK = 56


def fit_script_direction(
    embeddings,
    script_labels,
):
    probe = Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=PROBE_SEED,
                ),
            ),
        ]
    )

    probe.fit(
        embeddings,
        script_labels,
    )

    scaler = probe.named_steps[
        "scaler"
    ]

    classifier = probe.named_steps[
        "classifier"
    ]

    direction = (
        classifier.coef_[0]
        / scaler.scale_
    )

    return (
        probe,
        direction,
    )


def orthogonalize_direction(
    direction,
    basis,
):
    residual = direction.copy()

    for basis_direction in basis:
        residual = (
            residual
            - np.dot(
                residual,
                basis_direction,
            )
            * basis_direction
        )

    norm = np.linalg.norm(
        residual
    )

    if norm < 1e-12:
        raise RuntimeError(
            "Degenerate script-sensitive direction."
        )

    return (
        residual
        / norm
    )


def remove_direction(
    embeddings,
    direction,
):
    residual = (
        embeddings
        - np.outer(
            embeddings @ direction,
            direction,
        )
    )

    return normalize_rows(
        residual
    )


def build_fixed_script_subspace(
    development_embeddings,
    development_scripts,
    validation_embeddings,
):
    development_intervened = (
        development_embeddings.copy()
    )

    validation_intervened = (
        validation_embeddings.copy()
    )

    basis = []

    for dimension in range(
        1,
        FIXED_SCRIPT_SUBSPACE_RANK + 1,
    ):
        _, raw_direction = (
            fit_script_direction(
                development_intervened,
                development_scripts,
            )
        )

        direction = (
            orthogonalize_direction(
                raw_direction,
                basis,
            )
        )

        basis.append(
            direction
        )

        development_intervened = (
            remove_direction(
                development_intervened,
                direction,
            )
        )

        validation_intervened = (
            remove_direction(
                validation_intervened,
                direction,
            )
        )

    basis_matrix = np.stack(
        basis,
        axis=0,
    )

    return {
        "basis": basis_matrix,
        "development_intervened": (
            development_intervened
        ),
        "validation_intervened": (
            validation_intervened
        ),
    }


print(
    "Fixed script-sensitive intervention rank:",
    FIXED_SCRIPT_SUBSPACE_RANK,
)

print(
    "Representation stages:",
    len(
        representation_registry
    ),
)

Fixed script-sensitive intervention rank: 56
Representation stages: 12


In [13]:
script_intervention_registry = {}
script_intervention_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    for stage_name in [
        "frozen",
        "direct_lda",
        "pca225_lda",
    ]:
        representation = representation_registry[
            (
                model_name,
                stage_name,
            )
        ]

        print(
            "Running:",
            model_name,
            stage_name,
        )

        intervention = (
            build_fixed_script_subspace(
                representation[
                    "development_embeddings"
                ],
                representation[
                    "development_scripts"
                ],
                representation[
                    "validation_embeddings"
                ],
            )
        )

        residual_script_result = (
            evaluate_script_accessibility(
                intervention[
                    "development_intervened"
                ],
                representation[
                    "development_scripts"
                ],
                intervention[
                    "validation_intervened"
                ],
                representation[
                    "validation_scripts"
                ],
            )
        )

        original_writer_result = (
            evaluate_writer_verification(
                representation[
                    "validation_embeddings"
                ],
                representation[
                    "validation_filenames"
                ],
            )
        )

        intervened_writer_result = (
            evaluate_writer_verification(
                intervention[
                    "validation_intervened"
                ],
                representation[
                    "validation_filenames"
                ],
            )
        )

        maximum_removed_component = float(
            np.max(
                np.abs(
                    intervention[
                        "validation_intervened"
                    ]
                    @ intervention[
                        "basis"
                    ].T
                )
            )
        )

        basis_gram = (
            intervention[
                "basis"
            ]
            @ intervention[
                "basis"
            ].T
        )

        maximum_orthogonality_error = float(
            np.max(
                np.abs(
                    basis_gram
                    - np.eye(
                        FIXED_SCRIPT_SUBSPACE_RANK
                    )
                )
            )
        )

        script_intervention_registry[
            (
                model_name,
                stage_name,
            )
        ] = intervention

        script_intervention_rows.append(
            {
                "model": model_name,
                "stage": stage_name,
                "dimension": representation[
                    "validation_embeddings"
                ].shape[
                    1
                ],
                "removed_rank": (
                    FIXED_SCRIPT_SUBSPACE_RANK
                ),
                "original_script_auc": (
                    baseline_audit_df.loc[
                        (
                            baseline_audit_df[
                                "model"
                            ] == model_name
                        )
                        & (
                            baseline_audit_df[
                                "stage"
                            ] == stage_name
                        ),
                        "script_probe_auc",
                    ].iloc[0]
                ),
                "residual_script_auc": (
                    residual_script_result[
                        "script_auc"
                    ]
                ),
                "original_writer_auc": (
                    original_writer_result[
                        "overall_auc"
                    ]
                ),
                "intervened_writer_auc": (
                    intervened_writer_result[
                        "overall_auc"
                    ]
                ),
                "writer_auc_change": (
                    intervened_writer_result[
                        "overall_auc"
                    ]
                    - original_writer_result[
                        "overall_auc"
                    ]
                ),
                "original_cross_auc": (
                    original_writer_result[
                        "cross_macro_auc"
                    ]
                ),
                "intervened_cross_auc": (
                    intervened_writer_result[
                        "cross_macro_auc"
                    ]
                ),
                "cross_auc_change": (
                    intervened_writer_result[
                        "cross_macro_auc"
                    ]
                    - original_writer_result[
                        "cross_macro_auc"
                    ]
                ),
                "maximum_orthogonality_error": (
                    maximum_orthogonality_error
                ),
                "maximum_removed_component": (
                    maximum_removed_component
                ),
            }
        )


script_intervention_df = pd.DataFrame(
    script_intervention_rows
)

print()

print(
    script_intervention_df[
        [
            "model",
            "stage",
            "original_script_auc",
            "residual_script_auc",
            "original_writer_auc",
            "intervened_writer_auc",
            "writer_auc_change",
            "original_cross_auc",
            "intervened_cross_auc",
            "cross_auc_change",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Maximum orthogonality error:",
    script_intervention_df[
        "maximum_orthogonality_error"
    ].max(),
)

print(
    "Maximum remaining removed component:",
    script_intervention_df[
        "maximum_removed_component"
    ].max(),
)

Running: dinov2_s frozen
Running: dinov2_s direct_lda
Running: dinov2_s pca225_lda
Running: dinov2_b frozen
Running: dinov2_b direct_lda
Running: dinov2_b pca225_lda
Running: dinov2_l frozen
Running: dinov2_l direct_lda
Running: dinov2_l pca225_lda
Running: dinov2_g frozen
Running: dinov2_g direct_lda
Running: dinov2_g pca225_lda

   model      stage  original_script_auc  residual_script_auc  original_writer_auc  intervened_writer_auc  writer_auc_change  original_cross_auc  intervened_cross_auc  cross_auc_change
dinov2_s     frozen             1.000000             0.583307             0.701296               0.702844           0.001548            0.686559              0.693134          0.006575
dinov2_s direct_lda             0.638473             0.562500             0.846922               0.798761          -0.048160            0.804132              0.742085         -0.062048
dinov2_s pca225_lda             1.000000             0.507334             0.899126               0.841698       

In [14]:
def build_pair_intervention_effect(
    original_embeddings,
    intervened_embeddings,
    filenames,
):
    original_map = dict(
        zip(
            filenames,
            original_embeddings,
        )
    )

    intervened_map = dict(
        zip(
            filenames,
            intervened_embeddings,
        )
    )

    original_a = np.stack(
        [
            original_map[
                filename
            ]
            for filename in validation_pairs_df[
                "filename_a"
            ]
        ]
    )

    original_b = np.stack(
        [
            original_map[
                filename
            ]
            for filename in validation_pairs_df[
                "filename_b"
            ]
        ]
    )

    intervened_a = np.stack(
        [
            intervened_map[
                filename
            ]
            for filename in validation_pairs_df[
                "filename_a"
            ]
        ]
    )

    intervened_b = np.stack(
        [
            intervened_map[
                filename
            ]
            for filename in validation_pairs_df[
                "filename_b"
            ]
        ]
    )

    original_scores = np.sum(
        original_a
        * original_b,
        axis=1,
    )

    intervened_scores = np.sum(
        intervened_a
        * intervened_b,
        axis=1,
    )

    pair_df = (
        validation_pairs_df
        .copy()
        .reset_index(drop=True)
    )

    pair_df[
        "original_score"
    ] = original_scores

    pair_df[
        "intervened_score"
    ] = intervened_scores

    pair_df[
        "score_shift"
    ] = (
        intervened_scores
        - original_scores
    )

    pair_df[
        "script_reliance"
    ] = np.abs(
        pair_df[
            "score_shift"
        ]
    )

    pair_df[
        "condition"
    ] = [
        CONDITION_BY_PAGE_PAIR[
            (
                filename_page_id(
                    filename_a
                ),
                filename_page_id(
                    filename_b
                ),
            )
        ]
        for filename_a, filename_b in zip(
            pair_df[
                "filename_a"
            ],
            pair_df[
                "filename_b"
            ],
        )
    ]

    pair_df[
        "family"
    ] = np.where(
        pair_df[
            "condition"
        ].str.startswith(
            "cross_"
        ),
        "cross_script",
        "within_script",
    )

    return pair_df


pair_intervention_registry = {}
pair_reliance_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    for stage_name in [
        "frozen",
        "direct_lda",
        "pca225_lda",
    ]:
        representation = representation_registry[
            (
                model_name,
                stage_name,
            )
        ]

        intervention = script_intervention_registry[
            (
                model_name,
                stage_name,
            )
        ]

        pair_df = build_pair_intervention_effect(
            representation[
                "validation_embeddings"
            ],
            intervention[
                "validation_intervened"
            ],
            representation[
                "validation_filenames"
            ],
        )

        pair_intervention_registry[
            (
                model_name,
                stage_name,
            )
        ] = pair_df

        for family in [
            "within_script",
            "cross_script",
        ]:
            family_df = pair_df[
                pair_df[
                    "family"
                ] == family
            ]

            original_auc = roc_auc_score(
                family_df[
                    "pair_label"
                ],
                family_df[
                    "original_score"
                ],
            )

            intervened_auc = roc_auc_score(
                family_df[
                    "pair_label"
                ],
                family_df[
                    "intervened_score"
                ],
            )

            pair_reliance_rows.append(
                {
                    "model": model_name,
                    "stage": stage_name,
                    "family": family,
                    "mean_reliance": float(
                        family_df[
                            "script_reliance"
                        ].mean()
                    ),
                    "median_reliance": float(
                        family_df[
                            "script_reliance"
                        ].median()
                    ),
                    "p95_reliance": float(
                        family_df[
                            "script_reliance"
                        ].quantile(
                            0.95
                        )
                    ),
                    "original_auc": float(
                        original_auc
                    ),
                    "intervened_auc": float(
                        intervened_auc
                    ),
                    "auc_change": float(
                        intervened_auc
                        - original_auc
                    ),
                }
            )


pair_reliance_df = pd.DataFrame(
    pair_reliance_rows
)

print(
    pair_reliance_df
    .round(6)
    .to_string(
        index=False
    )
)

   model      stage        family  mean_reliance  median_reliance  p95_reliance  original_auc  intervened_auc  auc_change
dinov2_s     frozen within_script       0.016093         0.013364      0.039536      0.767484        0.770468    0.002984
dinov2_s     frozen  cross_script       0.012019         0.008935      0.033954      0.669466        0.675606    0.006140
dinov2_s direct_lda within_script       0.040210         0.032884      0.104023      0.930937        0.910327   -0.020610
dinov2_s direct_lda  cross_script       0.040147         0.033129      0.100703      0.804215        0.741924   -0.062291
dinov2_s pca225_lda within_script       0.041548         0.034571      0.106262      0.962451        0.930395   -0.032056
dinov2_s pca225_lda  cross_script       0.041751         0.034977      0.103481      0.867676        0.795815   -0.071861
dinov2_b     frozen within_script       0.011822         0.009979      0.028870      0.762403        0.747485   -0.014918
dinov2_b     frozen  cro

In [15]:
RANDOM_SUBSPACE_TRIALS = 20


def remove_subspace(
    embeddings,
    basis_matrix,
):
    projected = (
        embeddings
        - (
            embeddings
            @ basis_matrix.T
        )
        @ basis_matrix
    )

    return normalize_rows(
        projected
    )


random_control_rows = []

for model_index, model_name in enumerate(
    [
        "dinov2_s",
        "dinov2_b",
        "dinov2_l",
        "dinov2_g",
    ]
):
    for stage_index, stage_name in enumerate(
        [
            "frozen",
            "direct_lda",
            "pca225_lda",
        ]
    ):
        representation = representation_registry[
            (
                model_name,
                stage_name,
            )
        ]

        pair_df = pair_intervention_registry[
            (
                model_name,
                stage_name,
            )
        ]

        embeddings = representation[
            "validation_embeddings"
        ]

        dimension = embeddings.shape[
            1
        ]

        filenames = representation[
            "validation_filenames"
        ]

        filename_index = {
            filename: index
            for index, filename in enumerate(
                filenames
            )
        }

        pair_indices_a = np.array(
            [
                filename_index[
                    filename
                ]
                for filename in validation_pairs_df[
                    "filename_a"
                ]
            ]
        )

        pair_indices_b = np.array(
            [
                filename_index[
                    filename
                ]
                for filename in validation_pairs_df[
                    "filename_b"
                ]
            ]
        )

        original_scores = pair_df[
            "original_score"
        ].to_numpy()

        cross_mask = (
            pair_df[
                "family"
            ].to_numpy()
            == "cross_script"
        )

        cross_labels = pair_df.loc[
            cross_mask,
            "pair_label",
        ].to_numpy(
            dtype=np.int64
        )

        original_cross_auc = roc_auc_score(
            cross_labels,
            original_scores[
                cross_mask
            ],
        )

        for trial in range(
            RANDOM_SUBSPACE_TRIALS
        ):
            rng = np.random.default_rng(
                50000
                + 1000
                * model_index
                + 100
                * stage_index
                + trial
            )

            random_matrix = rng.normal(
                size=(
                    dimension,
                    FIXED_SCRIPT_SUBSPACE_RANK,
                )
            )

            random_basis, _ = np.linalg.qr(
                random_matrix
            )

            random_basis = (
                random_basis.T
            )

            random_embeddings = remove_subspace(
                embeddings,
                random_basis,
            )

            random_scores = np.sum(
                random_embeddings[
                    pair_indices_a
                ]
                * random_embeddings[
                    pair_indices_b
                ],
                axis=1,
            )

            random_reliance = np.abs(
                random_scores
                - original_scores
            )

            random_cross_auc = roc_auc_score(
                cross_labels,
                random_scores[
                    cross_mask
                ],
            )

            random_control_rows.append(
                {
                    "model": model_name,
                    "stage": stage_name,
                    "trial": trial,
                    "random_cross_reliance": float(
                        random_reliance[
                            cross_mask
                        ].mean()
                    ),
                    "random_cross_auc_change": float(
                        random_cross_auc
                        - original_cross_auc
                    ),
                }
            )


random_control_df = pd.DataFrame(
    random_control_rows
)

specificity_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    for stage_name in [
        "frozen",
        "direct_lda",
        "pca225_lda",
    ]:
        script_row = pair_reliance_df[
            (
                pair_reliance_df[
                    "model"
                ] == model_name
            )
            & (
                pair_reliance_df[
                    "stage"
                ] == stage_name
            )
            & (
                pair_reliance_df[
                    "family"
                ] == "cross_script"
            )
        ].iloc[
            0
        ]

        random_rows = random_control_df[
            (
                random_control_df[
                    "model"
                ] == model_name
            )
            & (
                random_control_df[
                    "stage"
                ] == stage_name
            )
        ]

        random_reliance_mean = random_rows[
            "random_cross_reliance"
        ].mean()

        random_reliance_std = random_rows[
            "random_cross_reliance"
        ].std(
            ddof=1
        )

        random_auc_change_mean = random_rows[
            "random_cross_auc_change"
        ].mean()

        random_auc_change_std = random_rows[
            "random_cross_auc_change"
        ].std(
            ddof=1
        )

        specificity_rows.append(
            {
                "model": model_name,
                "stage": stage_name,
                "script_probe_auc": float(
                    baseline_audit_df.loc[
                        (
                            baseline_audit_df[
                                "model"
                            ] == model_name
                        )
                        & (
                            baseline_audit_df[
                                "stage"
                            ] == stage_name
                        ),
                        "script_probe_auc",
                    ].iloc[
                        0
                    ]
                ),
                "residual_script_auc": float(
                    script_intervention_df.loc[
                        (
                            script_intervention_df[
                                "model"
                            ] == model_name
                        )
                        & (
                            script_intervention_df[
                                "stage"
                            ] == stage_name
                        ),
                        "residual_script_auc",
                    ].iloc[
                        0
                    ]
                ),
                "script_cross_reliance": float(
                    script_row[
                        "mean_reliance"
                    ]
                ),
                "random_cross_reliance_mean": float(
                    random_reliance_mean
                ),
                "random_cross_reliance_std": float(
                    random_reliance_std
                ),
                "script_to_random_ratio": float(
                    script_row[
                        "mean_reliance"
                    ]
                    / random_reliance_mean
                ),
                "script_cross_auc_change": float(
                    script_row[
                        "auc_change"
                    ]
                ),
                "random_cross_auc_change_mean": float(
                    random_auc_change_mean
                ),
                "random_cross_auc_change_std": float(
                    random_auc_change_std
                ),
            }
        )


specificity_df = pd.DataFrame(
    specificity_rows
)

print(
    specificity_df
    .round(6)
    .to_string(
        index=False
    )
)

   model      stage  script_probe_auc  residual_script_auc  script_cross_reliance  random_cross_reliance_mean  random_cross_reliance_std  script_to_random_ratio  script_cross_auc_change  random_cross_auc_change_mean  random_cross_auc_change_std
dinov2_s     frozen          1.000000             0.583307               0.012019                    0.004080                   0.001252                2.945490                 0.006140                     -0.000650                     0.002368
dinov2_s direct_lda          0.638473             0.562500               0.040147                    0.030360                   0.000623                1.322351                -0.062291                     -0.015514                     0.007349
dinov2_s pca225_lda          1.000000             0.507334               0.041751                    0.030483                   0.000542                1.369655                -0.071861                     -0.019317                     0.005605
dinov2_b     frozen 

In [16]:
random_extremeness_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    for stage_name in [
        "frozen",
        "direct_lda",
        "pca225_lda",
    ]:
        specificity_row = specificity_df[
            (
                specificity_df[
                    "model"
                ] == model_name
            )
            & (
                specificity_df[
                    "stage"
                ] == stage_name
            )
        ].iloc[
            0
        ]

        random_rows = random_control_df[
            (
                random_control_df[
                    "model"
                ] == model_name
            )
            & (
                random_control_df[
                    "stage"
                ] == stage_name
            )
        ]

        targeted_reliance = float(
            specificity_row[
                "script_cross_reliance"
            ]
        )

        targeted_abs_auc_effect = abs(
            float(
                specificity_row[
                    "script_cross_auc_change"
                ]
            )
        )

        random_reliance_values = random_rows[
            "random_cross_reliance"
        ].to_numpy()

        random_abs_auc_effects = np.abs(
            random_rows[
                "random_cross_auc_change"
            ].to_numpy()
        )

        reliance_exceed_count = int(
            (
                random_reliance_values
                >= targeted_reliance
            ).sum()
        )

        auc_effect_exceed_count = int(
            (
                random_abs_auc_effects
                >= targeted_abs_auc_effect
            ).sum()
        )

        reliance_z = (
            targeted_reliance
            - random_reliance_values.mean()
        ) / random_reliance_values.std(
            ddof=1
        )

        auc_effect_z = (
            targeted_abs_auc_effect
            - random_abs_auc_effects.mean()
        ) / random_abs_auc_effects.std(
            ddof=1
        )

        random_extremeness_rows.append(
            {
                "model": model_name,
                "stage": stage_name,
                "targeted_reliance": targeted_reliance,
                "random_reliance_mean": float(
                    random_reliance_values.mean()
                ),
                "reliance_ratio": float(
                    targeted_reliance
                    / random_reliance_values.mean()
                ),
                "random_reliance_exceed_count": (
                    reliance_exceed_count
                ),
                "reliance_z_vs_random": float(
                    reliance_z
                ),
                "targeted_abs_auc_effect": (
                    targeted_abs_auc_effect
                ),
                "random_abs_auc_effect_mean": float(
                    random_abs_auc_effects.mean()
                ),
                "random_auc_effect_exceed_count": (
                    auc_effect_exceed_count
                ),
                "auc_effect_z_vs_random": float(
                    auc_effect_z
                ),
            }
        )


random_extremeness_df = pd.DataFrame(
    random_extremeness_rows
)

print(
    random_extremeness_df[
        random_extremeness_df[
            "stage"
        ] == "pca225_lda"
    ]
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Full audit:"
)

print(
    random_extremeness_df
    .round(6)
    .to_string(
        index=False
    )
)

   model      stage  targeted_reliance  random_reliance_mean  reliance_ratio  random_reliance_exceed_count  reliance_z_vs_random  targeted_abs_auc_effect  random_abs_auc_effect_mean  random_auc_effect_exceed_count  auc_effect_z_vs_random
dinov2_s pca225_lda           0.041751              0.030483        1.369655                             0             20.792062                 0.071861                    0.019317                               0                9.374628
dinov2_b pca225_lda           0.044255              0.030167        1.467019                             0             18.625743                 0.059679                    0.016885                               0                4.923954
dinov2_l pca225_lda           0.042682              0.030614        1.394212                             0             21.174150                 0.047567                    0.014290                               0                4.166298
dinov2_g pca225_lda           0.038289          

In [17]:
random_script_probe_rows = []

pca_stage_index = 2

for model_index, model_name in enumerate(
    [
        "dinov2_s",
        "dinov2_b",
        "dinov2_l",
        "dinov2_g",
    ]
):
    representation = representation_registry[
        (
            model_name,
            "pca225_lda",
        )
    ]

    dimension = representation[
        "development_embeddings"
    ].shape[
        1
    ]

    for trial in range(
        RANDOM_SUBSPACE_TRIALS
    ):
        rng = np.random.default_rng(
            50000
            + 1000
            * model_index
            + 100
            * pca_stage_index
            + trial
        )

        random_matrix = rng.normal(
            size=(
                dimension,
                FIXED_SCRIPT_SUBSPACE_RANK,
            )
        )

        random_basis, _ = np.linalg.qr(
            random_matrix
        )

        random_basis = random_basis.T

        random_development = remove_subspace(
            representation[
                "development_embeddings"
            ],
            random_basis,
        )

        random_validation = remove_subspace(
            representation[
                "validation_embeddings"
            ],
            random_basis,
        )

        random_script_result = (
            evaluate_script_accessibility(
                random_development,
                representation[
                    "development_scripts"
                ],
                random_validation,
                representation[
                    "validation_scripts"
                ],
            )
        )

        random_script_probe_rows.append(
            {
                "model": model_name,
                "trial": trial,
                "random_residual_script_auc": (
                    random_script_result[
                        "script_auc"
                    ]
                ),
                "random_distance_from_chance": abs(
                    random_script_result[
                        "script_auc"
                    ]
                    - 0.5
                ),
            }
        )


random_script_probe_df = pd.DataFrame(
    random_script_probe_rows
)

script_removal_specificity_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    targeted_auc = float(
        script_intervention_df.loc[
            (
                script_intervention_df[
                    "model"
                ] == model_name
            )
            & (
                script_intervention_df[
                    "stage"
                ] == "pca225_lda"
            ),
            "residual_script_auc",
        ].iloc[
            0
        ]
    )

    random_rows = random_script_probe_df[
        random_script_probe_df[
            "model"
        ] == model_name
    ]

    random_distances = random_rows[
        "random_distance_from_chance"
    ].to_numpy()

    targeted_distance = abs(
        targeted_auc
        - 0.5
    )

    script_removal_specificity_rows.append(
        {
            "model": model_name,
            "targeted_residual_script_auc": (
                targeted_auc
            ),
            "targeted_distance_from_chance": (
                targeted_distance
            ),
            "random_residual_auc_mean": float(
                random_rows[
                    "random_residual_script_auc"
                ].mean()
            ),
            "random_residual_auc_std": float(
                random_rows[
                    "random_residual_script_auc"
                ].std(
                    ddof=1
                )
            ),
            "random_distance_from_chance_mean": float(
                random_distances.mean()
            ),
            "random_distance_from_chance_min": float(
                random_distances.min()
            ),
            "random_trials_closer_to_chance": int(
                (
                    random_distances
                    <= targeted_distance
                ).sum()
            ),
        }
    )


script_removal_specificity_df = pd.DataFrame(
    script_removal_specificity_rows
)

print(
    script_removal_specificity_df
    .round(6)
    .to_string(
        index=False
    )
)

   model  targeted_residual_script_auc  targeted_distance_from_chance  random_residual_auc_mean  random_residual_auc_std  random_distance_from_chance_mean  random_distance_from_chance_min  random_trials_closer_to_chance
dinov2_s                      0.507334                       0.007334                  0.953731                 0.015592                          0.453731                         0.428970                               0
dinov2_b                      0.523039                       0.023039                  0.962887                 0.018145                          0.462887                         0.400829                               0
dinov2_l                      0.410395                       0.089605                  0.975108                 0.012527                          0.475108                         0.445711                               0
dinov2_g                      0.561783                       0.061783                  0.970249                 0.011677

In [18]:
condition_reliance_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    pair_df = pair_intervention_registry[
        (
            model_name,
            "pca225_lda",
        )
    ]

    for condition, condition_df in pair_df.groupby(
        "condition"
    ):
        labels = condition_df[
            "pair_label"
        ].to_numpy(
            dtype=np.int64
        )

        original_auc = roc_auc_score(
            labels,
            condition_df[
                "original_score"
            ],
        )

        intervened_auc = roc_auc_score(
            labels,
            condition_df[
                "intervened_score"
            ],
        )

        condition_reliance_rows.append(
            {
                "model": model_name,
                "condition": condition,
                "family": (
                    "cross_script"
                    if condition.startswith(
                        "cross_"
                    )
                    else "within_script"
                ),
                "mean_reliance": float(
                    condition_df[
                        "script_reliance"
                    ].mean()
                ),
                "median_reliance": float(
                    condition_df[
                        "script_reliance"
                    ].median()
                ),
                "original_auc": float(
                    original_auc
                ),
                "intervened_auc": float(
                    intervened_auc
                ),
                "auc_change": float(
                    intervened_auc
                    - original_auc
                ),
            }
        )


condition_reliance_df = pd.DataFrame(
    condition_reliance_rows
)

print(
    condition_reliance_df[
        condition_reliance_df[
            "family"
        ] == "cross_script"
    ]
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Cross-script condition consistency:"
)

condition_consistency_df = (
    condition_reliance_df[
        condition_reliance_df[
            "family"
        ] == "cross_script"
    ]
    .groupby(
        "model"
    )
    .agg(
        cross_conditions=(
            "condition",
            "count",
        ),
        conditions_auc_decreased=(
            "auc_change",
            lambda values: int(
                (
                    values < 0
                ).sum()
            ),
        ),
        mean_auc_change=(
            "auc_change",
            "mean",
        ),
        min_auc_change=(
            "auc_change",
            "min",
        ),
        max_auc_change=(
            "auc_change",
            "max",
        ),
        mean_reliance=(
            "mean_reliance",
            "mean",
        ),
    )
    .reset_index()
)

print(
    condition_consistency_df
    .round(6)
    .to_string(
        index=False
    )
)

   model               condition       family  mean_reliance  median_reliance  original_auc  intervened_auc  auc_change
dinov2_s         cross_same_same cross_script       0.043045         0.036070      0.935540        0.875852   -0.059688
dinov2_s     cross_same_variable cross_script       0.042378         0.035473      0.894063        0.796336   -0.097727
dinov2_s     cross_variable_same cross_script       0.040936         0.033974      0.817544        0.739877   -0.077667
dinov2_s cross_variable_variable cross_script       0.040643         0.034230      0.824954        0.770020   -0.054934
dinov2_b         cross_same_same cross_script       0.042933         0.035137      0.918512        0.871226   -0.047287
dinov2_b     cross_same_variable cross_script       0.045449         0.037342      0.913062        0.870466   -0.042596
dinov2_b     cross_variable_same cross_script       0.042643         0.035104      0.866019        0.772948   -0.093072
dinov2_b cross_variable_variable cross_s

In [19]:
text_values = sorted(
    development_df[
        "same_text"
    ]
    .astype(str)
    .unique()
    .tolist()
)

assert len(
    text_values
) == 2

TEXT_TO_LABEL = {
    value: index
    for index, value in enumerate(
        text_values
    )
}

development_text_map = {
    filename: TEXT_TO_LABEL[
        str(
            same_text
        )
    ]
    for filename, same_text in zip(
        development_df[
            "filename"
        ],
        development_df[
            "same_text"
        ],
    )
}

validation_text_map = {
    filename: TEXT_TO_LABEL[
        str(
            same_text
        )
    ]
    for filename, same_text in zip(
        validation_df[
            "filename"
        ],
        validation_df[
            "same_text"
        ],
    )
}


def evaluate_binary_accessibility(
    development_embeddings,
    development_labels,
    validation_embeddings,
    validation_labels,
):
    probe = Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=PROBE_SEED,
                ),
            ),
        ]
    )

    probe.fit(
        development_embeddings,
        development_labels,
    )

    probabilities = probe.predict_proba(
        validation_embeddings
    )[
        :,
        1
    ]

    predictions = probe.predict(
        validation_embeddings
    )

    return {
        "auc": float(
            roc_auc_score(
                validation_labels,
                probabilities,
            )
        ),
        "accuracy": float(
            accuracy_score(
                validation_labels,
                predictions,
            )
        ),
    }


content_control_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    representation = representation_registry[
        (
            model_name,
            "pca225_lda",
        )
    ]

    intervention = script_intervention_registry[
        (
            model_name,
            "pca225_lda",
        )
    ]

    development_filenames = representation[
        "development_filenames"
    ]

    validation_filenames = representation[
        "validation_filenames"
    ]

    development_text_labels = np.array(
        [
            development_text_map[
                filename
            ]
            for filename in development_filenames
        ],
        dtype=np.int64,
    )

    validation_text_labels = np.array(
        [
            validation_text_map[
                filename
            ]
            for filename in validation_filenames
        ],
        dtype=np.int64,
    )

    original_text_result = (
        evaluate_binary_accessibility(
            representation[
                "development_embeddings"
            ],
            development_text_labels,
            representation[
                "validation_embeddings"
            ],
            validation_text_labels,
        )
    )

    intervened_text_result = (
        evaluate_binary_accessibility(
            intervention[
                "development_intervened"
            ],
            development_text_labels,
            intervention[
                "validation_intervened"
            ],
            validation_text_labels,
        )
    )

    original_script_result = (
        evaluate_script_accessibility(
            representation[
                "development_embeddings"
            ],
            representation[
                "development_scripts"
            ],
            representation[
                "validation_embeddings"
            ],
            representation[
                "validation_scripts"
            ],
        )
    )

    intervened_script_result = (
        evaluate_script_accessibility(
            intervention[
                "development_intervened"
            ],
            representation[
                "development_scripts"
            ],
            intervention[
                "validation_intervened"
            ],
            representation[
                "validation_scripts"
            ],
        )
    )

    content_control_rows.append(
        {
            "model": model_name,
            "original_script_auc": (
                original_script_result[
                    "script_auc"
                ]
            ),
            "intervened_script_auc": (
                intervened_script_result[
                    "script_auc"
                ]
            ),
            "script_auc_change": (
                intervened_script_result[
                    "script_auc"
                ]
                - original_script_result[
                    "script_auc"
                ]
            ),
            "original_text_regime_auc": (
                original_text_result[
                    "auc"
                ]
            ),
            "intervened_text_regime_auc": (
                intervened_text_result[
                    "auc"
                ]
            ),
            "text_regime_auc_change": (
                intervened_text_result[
                    "auc"
                ]
                - original_text_result[
                    "auc"
                ]
            ),
        }
    )


content_control_df = pd.DataFrame(
    content_control_rows
)

print(
    "Text-regime mapping:",
    TEXT_TO_LABEL,
)

print()

print(
    development_df[
        [
            "page_id",
            "language",
            "same_text",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "page_id"
    )
    .to_string(
        index=False
    )
)

print()

print(
    content_control_df
    .round(6)
    .to_string(
        index=False
    )
)

Text-regime mapping: {'0': 0, '1': 1}

 page_id language  same_text
       1   Arabic          0
       2   Arabic          1
       3  English          0
       4  English          1

   model  original_script_auc  intervened_script_auc  script_auc_change  original_text_regime_auc  intervened_text_regime_auc  text_regime_auc_change
dinov2_s              1.00000               0.507334          -0.492666                  0.999681                    0.949139               -0.050542
dinov2_b              1.00000               0.523039          -0.476961                  0.996971                    0.949936               -0.047034
dinov2_l              1.00000               0.410395          -0.589605                  0.998246                    0.942363               -0.055883
dinov2_g              0.99992               0.561783          -0.438138                  0.994739                    0.958227               -0.036511


In [20]:
DOSE_RESPONSE_RANKS = [
    0,
    1,
    2,
    4,
    8,
    16,
    32,
    56,
]


def cross_pooled_auc(
    pair_df,
    score_column,
):
    cross_df = pair_df[
        pair_df[
            "family"
        ] == "cross_script"
    ]

    return float(
        roc_auc_score(
            cross_df[
                "pair_label"
            ],
            cross_df[
                score_column
            ],
        )
    )


dose_response_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    representation = representation_registry[
        (
            model_name,
            "pca225_lda",
        )
    ]

    full_basis = script_intervention_registry[
        (
            model_name,
            "pca225_lda",
        )
    ][
        "basis"
    ]

    development_text_labels = np.array(
        [
            development_text_map[
                filename
            ]
            for filename in representation[
                "development_filenames"
            ]
        ],
        dtype=np.int64,
    )

    validation_text_labels = np.array(
        [
            validation_text_map[
                filename
            ]
            for filename in representation[
                "validation_filenames"
            ]
        ],
        dtype=np.int64,
    )

    original_pair_df = build_pair_intervention_effect(
        representation[
            "validation_embeddings"
        ],
        representation[
            "validation_embeddings"
        ],
        representation[
            "validation_filenames"
        ],
    )

    original_cross_pooled_auc = (
        cross_pooled_auc(
            original_pair_df,
            "original_score",
        )
    )

    for removed_rank in DOSE_RESPONSE_RANKS:
        if removed_rank == 0:
            development_current = representation[
                "development_embeddings"
            ]

            validation_current = representation[
                "validation_embeddings"
            ]
        else:
            current_basis = full_basis[
                :removed_rank
            ]

            development_current = remove_subspace(
                representation[
                    "development_embeddings"
                ],
                current_basis,
            )

            validation_current = remove_subspace(
                representation[
                    "validation_embeddings"
                ],
                current_basis,
            )

        script_result = (
            evaluate_script_accessibility(
                development_current,
                representation[
                    "development_scripts"
                ],
                validation_current,
                representation[
                    "validation_scripts"
                ],
            )
        )

        text_result = (
            evaluate_binary_accessibility(
                development_current,
                development_text_labels,
                validation_current,
                validation_text_labels,
            )
        )

        writer_result = (
            evaluate_writer_verification(
                validation_current,
                representation[
                    "validation_filenames"
                ],
            )
        )

        pair_df = build_pair_intervention_effect(
            representation[
                "validation_embeddings"
            ],
            validation_current,
            representation[
                "validation_filenames"
            ],
        )

        cross_df = pair_df[
            pair_df[
                "family"
            ] == "cross_script"
        ]

        current_cross_pooled_auc = (
            cross_pooled_auc(
                pair_df,
                "intervened_score",
            )
        )

        dose_response_rows.append(
            {
                "model": model_name,
                "removed_rank": removed_rank,
                "script_auc": script_result[
                    "script_auc"
                ],
                "script_distance_from_chance": abs(
                    script_result[
                        "script_auc"
                    ]
                    - 0.5
                ),
                "text_regime_auc": text_result[
                    "auc"
                ],
                "overall_writer_auc": writer_result[
                    "overall_auc"
                ],
                "cross_macro_auc": writer_result[
                    "cross_macro_auc"
                ],
                "cross_pooled_auc": (
                    current_cross_pooled_auc
                ),
                "cross_pooled_auc_change": (
                    current_cross_pooled_auc
                    - original_cross_pooled_auc
                ),
                "mean_cross_reliance": float(
                    cross_df[
                        "script_reliance"
                    ].mean()
                ),
            }
        )


dose_response_df = pd.DataFrame(
    dose_response_rows
)

print(
    dose_response_df
    .round(6)
    .to_string(
        index=False
    )
)

   model  removed_rank  script_auc  script_distance_from_chance  text_regime_auc  overall_writer_auc  cross_macro_auc  cross_pooled_auc  cross_pooled_auc_change  mean_cross_reliance
dinov2_s             0    1.000000                     0.500000         0.999681            0.899126         0.868025          0.867676                 0.000000             0.000000
dinov2_s             1    0.716438                     0.216438         0.999681            0.900065         0.868082          0.867759                 0.000083             0.002226
dinov2_s             2    0.573661                     0.073661         0.999681            0.900193         0.868248          0.867852                 0.000176             0.002979
dinov2_s             4    0.515067                     0.015067         0.999203            0.900524         0.868509          0.868099                 0.000423             0.005380
dinov2_s             8    0.506138                     0.006138         0.999522          

In [21]:
RANDOM_DOSE_TRIALS = 20

random_dose_rows = []

for model_index, model_name in enumerate(
    [
        "dinov2_s",
        "dinov2_b",
        "dinov2_l",
        "dinov2_g",
    ]
):
    representation = representation_registry[
        (
            model_name,
            "pca225_lda",
        )
    ]

    embeddings = representation[
        "validation_embeddings"
    ]

    filenames = representation[
        "validation_filenames"
    ]

    dimension = embeddings.shape[
        1
    ]

    original_pair_df = build_pair_intervention_effect(
        embeddings,
        embeddings,
        filenames,
    )

    original_scores = original_pair_df[
        "original_score"
    ].to_numpy()

    cross_mask = (
        original_pair_df[
            "family"
        ].to_numpy()
        == "cross_script"
    )

    cross_labels = original_pair_df.loc[
        cross_mask,
        "pair_label",
    ].to_numpy(
        dtype=np.int64
    )

    original_cross_auc = roc_auc_score(
        cross_labels,
        original_scores[
            cross_mask
        ],
    )

    filename_index = {
        filename: index
        for index, filename in enumerate(
            filenames
        )
    }

    pair_indices_a = np.array(
        [
            filename_index[
                filename
            ]
            for filename in validation_pairs_df[
                "filename_a"
            ]
        ]
    )

    pair_indices_b = np.array(
        [
            filename_index[
                filename
            ]
            for filename in validation_pairs_df[
                "filename_b"
            ]
        ]
    )

    for trial in range(
        RANDOM_DOSE_TRIALS
    ):
        rng = np.random.default_rng(
            50000
            + 1000
            * model_index
            + 200
            + trial
        )

        random_matrix = rng.normal(
            size=(
                dimension,
                FIXED_SCRIPT_SUBSPACE_RANK,
            )
        )

        full_random_basis, _ = np.linalg.qr(
            random_matrix
        )

        full_random_basis = (
            full_random_basis.T
        )

        for removed_rank in [
            1,
            2,
            4,
            8,
            16,
            32,
            56,
        ]:
            random_embeddings = remove_subspace(
                embeddings,
                full_random_basis[
                    :removed_rank
                ],
            )

            random_scores = np.sum(
                random_embeddings[
                    pair_indices_a
                ]
                * random_embeddings[
                    pair_indices_b
                ],
                axis=1,
            )

            random_cross_auc = roc_auc_score(
                cross_labels,
                random_scores[
                    cross_mask
                ],
            )

            random_dose_rows.append(
                {
                    "model": model_name,
                    "removed_rank": removed_rank,
                    "trial": trial,
                    "random_cross_reliance": float(
                        np.abs(
                            random_scores[
                                cross_mask
                            ]
                            - original_scores[
                                cross_mask
                            ]
                        ).mean()
                    ),
                    "random_cross_auc_change": float(
                        random_cross_auc
                        - original_cross_auc
                    ),
                }
            )


random_dose_df = pd.DataFrame(
    random_dose_rows
)

random_dose_summary_df = (
    random_dose_df
    .groupby(
        [
            "model",
            "removed_rank",
        ]
    )
    .agg(
        random_reliance_mean=(
            "random_cross_reliance",
            "mean",
        ),
        random_reliance_std=(
            "random_cross_reliance",
            "std",
        ),
        random_auc_change_mean=(
            "random_cross_auc_change",
            "mean",
        ),
        random_auc_change_std=(
            "random_cross_auc_change",
            "std",
        ),
    )
    .reset_index()
)

targeted_nonzero_df = dose_response_df[
    dose_response_df[
        "removed_rank"
    ] > 0
][
    [
        "model",
        "removed_rank",
        "script_auc",
        "text_regime_auc",
        "cross_pooled_auc_change",
        "mean_cross_reliance",
    ]
]

dose_specificity_df = targeted_nonzero_df.merge(
    random_dose_summary_df,
    on=[
        "model",
        "removed_rank",
    ],
    how="left",
)

dose_specificity_df[
    "reliance_ratio"
] = (
    dose_specificity_df[
        "mean_cross_reliance"
    ]
    / dose_specificity_df[
        "random_reliance_mean"
    ]
)

dose_specificity_df[
    "targeted_abs_auc_effect"
] = np.abs(
    dose_specificity_df[
        "cross_pooled_auc_change"
    ]
)

dose_specificity_df[
    "random_abs_auc_effect_approx"
] = np.abs(
    dose_specificity_df[
        "random_auc_change_mean"
    ]
)

print(
    dose_specificity_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

rank56_previous = specificity_df[
    specificity_df[
        "stage"
    ] == "pca225_lda"
][
    [
        "model",
        "random_cross_reliance_mean",
    ]
].rename(
    columns={
        "random_cross_reliance_mean": (
            "previous_rank56_random_reliance"
        )
    }
)

rank56_current = random_dose_summary_df[
    random_dose_summary_df[
        "removed_rank"
    ] == 56
][
    [
        "model",
        "random_reliance_mean",
    ]
]

rank56_check = rank56_previous.merge(
    rank56_current,
    on="model",
)

rank56_check[
    "difference"
] = (
    rank56_check[
        "random_reliance_mean"
    ]
    - rank56_check[
        "previous_rank56_random_reliance"
    ]
)

print(
    "Rank-56 random-control consistency:"
)

print(
    rank56_check
    .round(6)
    .to_string(
        index=False
    )
)

   model  removed_rank  script_auc  text_regime_auc  cross_pooled_auc_change  mean_cross_reliance  random_reliance_mean  random_reliance_std  random_auc_change_mean  random_auc_change_std  reliance_ratio  targeted_abs_auc_effect  random_abs_auc_effect_approx
dinov2_s             1    0.716438         0.999681                 0.000083             0.002226              0.002809             0.000280               -0.000450               0.000677        0.792557                 0.000083                      0.000450
dinov2_s             2    0.573661         0.999681                 0.000176             0.002979              0.004489             0.000363               -0.000722               0.000828        0.663754                 0.000176                      0.000722
dinov2_s             4    0.515067         0.999203                 0.000423             0.005380              0.006706             0.000459               -0.001299               0.001159        0.802327                 0.0

In [22]:
SPLIT_SEED = 42
AUC_NULL_Z = 1.96

reference_representation = representation_registry[
    (
        "dinov2_s",
        "pca225_lda",
    )
]

development_writers_reference = (
    reference_representation[
        "development_writers"
    ]
)

development_scripts_reference = (
    reference_representation[
        "development_scripts"
    ]
)

unique_development_writers = np.unique(
    development_writers_reference
)

rng = np.random.default_rng(
    SPLIT_SEED
)

shuffled_writers = rng.permutation(
    unique_development_writers
)

monitor_writer_count = int(
    round(
        0.20
        * len(
            shuffled_writers
        )
    )
)

monitor_writers = shuffled_writers[
    :monitor_writer_count
]

subspace_train_writers = shuffled_writers[
    monitor_writer_count:
]

reference_monitor_mask = np.isin(
    development_writers_reference,
    monitor_writers,
)

monitor_script_counts = np.bincount(
    development_scripts_reference[
        reference_monitor_mask
    ]
)

monitor_negative_count = int(
    monitor_script_counts[
        0
    ]
)

monitor_positive_count = int(
    monitor_script_counts[
        1
    ]
)

auc_null_se = np.sqrt(
    (
        monitor_negative_count
        + monitor_positive_count
        + 1
    )
    / (
        12
        * monitor_negative_count
        * monitor_positive_count
    )
)

AUC_CHANCE_UPPER = (
    0.5
    + AUC_NULL_Z
    * auc_null_se
)

development_rank_rows = []
development_rank_selection_rows = []

for model_name in [
    "dinov2_s",
    "dinov2_b",
    "dinov2_l",
    "dinov2_g",
]:
    representation = representation_registry[
        (
            model_name,
            "pca225_lda",
        )
    ]

    writers = representation[
        "development_writers"
    ]

    scripts = representation[
        "development_scripts"
    ]

    train_mask = np.isin(
        writers,
        subspace_train_writers,
    )

    monitor_mask = np.isin(
        writers,
        monitor_writers,
    )

    train_embeddings = representation[
        "development_embeddings"
    ][
        train_mask
    ].copy()

    monitor_embeddings = representation[
        "development_embeddings"
    ][
        monitor_mask
    ].copy()

    train_scripts = scripts[
        train_mask
    ]

    monitor_scripts = scripts[
        monitor_mask
    ]

    initial_result = (
        evaluate_script_accessibility(
            train_embeddings,
            train_scripts,
            monitor_embeddings,
            monitor_scripts,
        )
    )

    development_rank_rows.append(
        {
            "model": model_name,
            "removed_rank": 0,
            "monitor_script_auc": (
                initial_result[
                    "script_auc"
                ]
            ),
            "effective_monitor_auc": max(
                initial_result[
                    "script_auc"
                ],
                1.0
                - initial_result[
                    "script_auc"
                ],
            ),
        }
    )

    basis = []

    for removed_rank in range(
        1,
        FIXED_SCRIPT_SUBSPACE_RANK + 1,
    ):
        _, raw_direction = (
            fit_script_direction(
                train_embeddings,
                train_scripts,
            )
        )

        direction = (
            orthogonalize_direction(
                raw_direction,
                basis,
            )
        )

        basis.append(
            direction
        )

        train_embeddings = (
            remove_direction(
                train_embeddings,
                direction,
            )
        )

        monitor_embeddings = (
            remove_direction(
                monitor_embeddings,
                direction,
            )
        )

        residual_result = (
            evaluate_script_accessibility(
                train_embeddings,
                train_scripts,
                monitor_embeddings,
                monitor_scripts,
            )
        )

        monitor_auc = residual_result[
            "script_auc"
        ]

        effective_auc = max(
            monitor_auc,
            1.0
            - monitor_auc,
        )

        development_rank_rows.append(
            {
                "model": model_name,
                "removed_rank": removed_rank,
                "monitor_script_auc": (
                    monitor_auc
                ),
                "effective_monitor_auc": (
                    effective_auc
                ),
            }
        )

    model_rank_df = pd.DataFrame(
        [
            row
            for row in development_rank_rows
            if row[
                "model"
            ] == model_name
        ]
    )

    eligible_rows = model_rank_df[
        (
            model_rank_df[
                "removed_rank"
            ] > 0
        )
        & (
            model_rank_df[
                "effective_monitor_auc"
            ]
            <= AUC_CHANCE_UPPER
        )
    ]

    if len(
        eligible_rows
    ) > 0:
        selected_rank = int(
            eligible_rows.iloc[
                0
            ][
                "removed_rank"
            ]
        )

        selection_reason = (
            "first_rank_inside_null_auc_band"
        )
    else:
        nonzero_rows = model_rank_df[
            model_rank_df[
                "removed_rank"
            ] > 0
        ]

        selected_rank = int(
            nonzero_rows.loc[
                nonzero_rows[
                    "effective_monitor_auc"
                ].idxmin(),
                "removed_rank",
            ]
        )

        selection_reason = (
            "closest_rank_to_chance"
        )

    selected_row = model_rank_df[
        model_rank_df[
            "removed_rank"
        ] == selected_rank
    ].iloc[
        0
    ]

    development_rank_selection_rows.append(
        {
            "model": model_name,
            "selected_rank": (
                selected_rank
            ),
            "monitor_script_auc": float(
                selected_row[
                    "monitor_script_auc"
                ]
            ),
            "effective_monitor_auc": float(
                selected_row[
                    "effective_monitor_auc"
                ]
            ),
            "selection_reason": (
                selection_reason
            ),
        }
    )


development_rank_trajectory_df = pd.DataFrame(
    development_rank_rows
)

development_rank_selection_df = pd.DataFrame(
    development_rank_selection_rows
)

print(
    "Subspace-train writers:",
    len(
        subspace_train_writers
    ),
)

print(
    "Monitor writers:",
    len(
        monitor_writers
    ),
)

print(
    "Writer overlap:",
    len(
        set(
            subspace_train_writers
        ).intersection(
            set(
                monitor_writers
            )
        )
    ),
)

print(
    "Monitor script counts:",
    monitor_script_counts.tolist(),
)

print(
    "95% null AUC upper band:",
    AUC_CHANCE_UPPER,
)

print()

print(
    development_rank_selection_df
    .round(6)
    .to_string(
        index=False
    )
)

Subspace-train writers: 181
Monitor writers: 45
Writer overlap: 0
Monitor script counts: [90, 90]
95% null AUC upper band: 0.5845789377316772

   model  selected_rank  monitor_script_auc  effective_monitor_auc                selection_reason
dinov2_s              3            0.512099               0.512099 first_rank_inside_null_auc_band
dinov2_b              3            0.416173               0.583827 first_rank_inside_null_auc_band
dinov2_l              3            0.431358               0.568642 first_rank_inside_null_auc_band
dinov2_g              1            0.465556               0.534444 first_rank_inside_null_auc_band


In [23]:
def build_script_subspace_at_rank(
    development_embeddings,
    development_scripts,
    validation_embeddings,
    rank,
):
    development_current = (
        development_embeddings.copy()
    )

    validation_current = (
        validation_embeddings.copy()
    )

    basis = []

    for _ in range(
        rank
    ):
        _, raw_direction = (
            fit_script_direction(
                development_current,
                development_scripts,
            )
        )

        direction = (
            orthogonalize_direction(
                raw_direction,
                basis,
            )
        )

        basis.append(
            direction
        )

        development_current = (
            remove_direction(
                development_current,
                direction,
            )
        )

        validation_current = (
            remove_direction(
                validation_current,
                direction,
            )
        )

    return {
        "basis": np.stack(
            basis,
            axis=0,
        ),
        "development": (
            development_current
        ),
        "validation": (
            validation_current
        ),
    }


development_selected_validation_rows = []

for model_index, model_name in enumerate(
    [
        "dinov2_s",
        "dinov2_b",
        "dinov2_l",
        "dinov2_g",
    ]
):
    selected_rank = int(
        development_rank_selection_df.loc[
            development_rank_selection_df[
                "model"
            ] == model_name,
            "selected_rank",
        ].iloc[
            0
        ]
    )

    representation = representation_registry[
        (
            model_name,
            "pca225_lda",
        )
    ]

    intervention = build_script_subspace_at_rank(
        representation[
            "development_embeddings"
        ],
        representation[
            "development_scripts"
        ],
        representation[
            "validation_embeddings"
        ],
        selected_rank,
    )

    residual_script = (
        evaluate_script_accessibility(
            intervention[
                "development"
            ],
            representation[
                "development_scripts"
            ],
            intervention[
                "validation"
            ],
            representation[
                "validation_scripts"
            ],
        )
    )

    original_writer = (
        evaluate_writer_verification(
            representation[
                "validation_embeddings"
            ],
            representation[
                "validation_filenames"
            ],
        )
    )

    intervened_writer = (
        evaluate_writer_verification(
            intervention[
                "validation"
            ],
            representation[
                "validation_filenames"
            ],
        )
    )

    pair_df = build_pair_intervention_effect(
        representation[
            "validation_embeddings"
        ],
        intervention[
            "validation"
        ],
        representation[
            "validation_filenames"
        ],
    )

    cross_df = pair_df[
        pair_df[
            "family"
        ] == "cross_script"
    ]

    targeted_reliance = float(
        cross_df[
            "script_reliance"
        ].mean()
    )

    targeted_original_cross_auc = roc_auc_score(
        cross_df[
            "pair_label"
        ],
        cross_df[
            "original_score"
        ],
    )

    targeted_intervened_cross_auc = roc_auc_score(
        cross_df[
            "pair_label"
        ],
        cross_df[
            "intervened_score"
        ],
    )

    random_reliance_values = []
    random_auc_changes = []
    random_residual_script_aucs = []

    dimension = representation[
        "validation_embeddings"
    ].shape[
        1
    ]

    for trial in range(
        RANDOM_SUBSPACE_TRIALS
    ):
        rng = np.random.default_rng(
            80000
            + 1000
            * model_index
            + trial
        )

        random_matrix = rng.normal(
            size=(
                dimension,
                selected_rank,
            )
        )

        random_basis, _ = np.linalg.qr(
            random_matrix
        )

        random_basis = random_basis.T

        random_development = remove_subspace(
            representation[
                "development_embeddings"
            ],
            random_basis,
        )

        random_validation = remove_subspace(
            representation[
                "validation_embeddings"
            ],
            random_basis,
        )

        random_script = (
            evaluate_script_accessibility(
                random_development,
                representation[
                    "development_scripts"
                ],
                random_validation,
                representation[
                    "validation_scripts"
                ],
            )
        )

        random_pair_df = (
            build_pair_intervention_effect(
                representation[
                    "validation_embeddings"
                ],
                random_validation,
                representation[
                    "validation_filenames"
                ],
            )
        )

        random_cross_df = random_pair_df[
            random_pair_df[
                "family"
            ] == "cross_script"
        ]

        random_cross_auc = roc_auc_score(
            random_cross_df[
                "pair_label"
            ],
            random_cross_df[
                "intervened_score"
            ],
        )

        random_reliance_values.append(
            random_cross_df[
                "script_reliance"
            ].mean()
        )

        random_auc_changes.append(
            random_cross_auc
            - targeted_original_cross_auc
        )

        random_residual_script_aucs.append(
            random_script[
                "script_auc"
            ]
        )

    random_reliance_values = np.array(
        random_reliance_values
    )

    random_auc_changes = np.array(
        random_auc_changes
    )

    random_residual_script_aucs = np.array(
        random_residual_script_aucs
    )

    development_selected_validation_rows.append(
        {
            "model": model_name,
            "selected_rank": selected_rank,
            "validation_residual_script_auc": (
                residual_script[
                    "script_auc"
                ]
            ),
            "original_writer_auc": (
                original_writer[
                    "overall_auc"
                ]
            ),
            "intervened_writer_auc": (
                intervened_writer[
                    "overall_auc"
                ]
            ),
            "writer_auc_change": (
                intervened_writer[
                    "overall_auc"
                ]
                - original_writer[
                    "overall_auc"
                ]
            ),
            "original_cross_macro_auc": (
                original_writer[
                    "cross_macro_auc"
                ]
            ),
            "intervened_cross_macro_auc": (
                intervened_writer[
                    "cross_macro_auc"
                ]
            ),
            "cross_macro_auc_change": (
                intervened_writer[
                    "cross_macro_auc"
                ]
                - original_writer[
                    "cross_macro_auc"
                ]
            ),
            "cross_pooled_auc_change": (
                targeted_intervened_cross_auc
                - targeted_original_cross_auc
            ),
            "targeted_cross_reliance": (
                targeted_reliance
            ),
            "random_cross_reliance_mean": float(
                random_reliance_values.mean()
            ),
            "targeted_to_random_reliance_ratio": float(
                targeted_reliance
                / random_reliance_values.mean()
            ),
            "random_cross_auc_change_mean": float(
                random_auc_changes.mean()
            ),
            "random_residual_script_auc_mean": float(
                random_residual_script_aucs.mean()
            ),
        }
    )


development_selected_validation_df = (
    pd.DataFrame(
        development_selected_validation_rows
    )
)

print(
    development_selected_validation_df
    .round(6)
    .to_string(
        index=False
    )
)

   model  selected_rank  validation_residual_script_auc  original_writer_auc  intervened_writer_auc  writer_auc_change  original_cross_macro_auc  intervened_cross_macro_auc  cross_macro_auc_change  cross_pooled_auc_change  targeted_cross_reliance  random_cross_reliance_mean  targeted_to_random_reliance_ratio  random_cross_auc_change_mean  random_residual_script_auc_mean
dinov2_s              3                        0.499681             0.899126               0.900301           0.001175                  0.868025                    0.868331                0.000306                 0.000248                 0.003686                    0.005623                           0.655472                     -0.001085                         0.999821
dinov2_b              3                        0.573262             0.914989               0.916423           0.001434                  0.890048                    0.890528                0.000480                 0.000442                 0.003307        

In [24]:
NOTEBOOK21_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "foundation_script_reliance_audit"
)

NOTEBOOK21_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

report_tables = {
    "representation_inventory.csv": (
        representation_inventory_df
    ),
    "baseline_accessibility_audit.csv": (
        baseline_audit_df
    ),
    "fixed56_intervention_summary.csv": (
        script_intervention_df
    ),
    "fixed56_pair_reliance.csv": (
        pair_reliance_df
    ),
    "fixed56_random_subspace_trials.csv": (
        random_control_df
    ),
    "fixed56_script_specificity.csv": (
        specificity_df
    ),
    "random_control_extremeness.csv": (
        random_extremeness_df
    ),
    "random_script_probe_control.csv": (
        random_script_probe_df
    ),
    "script_removal_specificity.csv": (
        script_removal_specificity_df
    ),
    "condition_level_reliance.csv": (
        condition_reliance_df
    ),
    "condition_consistency.csv": (
        condition_consistency_df
    ),
    "text_regime_control.csv": (
        content_control_df
    ),
    "dose_response.csv": (
        dose_response_df
    ),
    "random_dose_trials.csv": (
        random_dose_df
    ),
    "dose_specificity.csv": (
        dose_specificity_df
    ),
    "development_rank_trajectory.csv": (
        development_rank_trajectory_df
    ),
    "development_rank_selection.csv": (
        development_rank_selection_df
    ),
    "development_selected_validation.csv": (
        development_selected_validation_df
    ),
}

saved_report_paths = []

for filename, dataframe in (
    report_tables.items()
):
    output_path = (
        NOTEBOOK21_REPORT_DIR
        / filename
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )

    saved_report_paths.append(
        output_path
    )

print(
    "Saved tables:",
    len(
        saved_report_paths
    ),
)

for path in saved_report_paths:
    print(
        "-",
        path.relative_to(
            PROJECT_ROOT
        ),
    )

Saved tables: 18
- reports\foundation_script_reliance_audit\representation_inventory.csv
- reports\foundation_script_reliance_audit\baseline_accessibility_audit.csv
- reports\foundation_script_reliance_audit\fixed56_intervention_summary.csv
- reports\foundation_script_reliance_audit\fixed56_pair_reliance.csv
- reports\foundation_script_reliance_audit\fixed56_random_subspace_trials.csv
- reports\foundation_script_reliance_audit\fixed56_script_specificity.csv
- reports\foundation_script_reliance_audit\random_control_extremeness.csv
- reports\foundation_script_reliance_audit\random_script_probe_control.csv
- reports\foundation_script_reliance_audit\script_removal_specificity.csv
- reports\foundation_script_reliance_audit\condition_level_reliance.csv
- reports\foundation_script_reliance_audit\condition_consistency.csv
- reports\foundation_script_reliance_audit\text_regime_control.csv
- reports\foundation_script_reliance_audit\dose_response.csv
- reports\foundation_script_reliance_audit\ran

In [25]:
confirmatory_rows = {}

for _, row in (
    development_selected_validation_df
    .iterrows()
):
    model_name = row[
        "model"
    ]

    confirmatory_rows[
        model_name
    ] = {
        "development_selected_rank": int(
            row[
                "selected_rank"
            ]
        ),
        "validation_residual_script_auc": float(
            row[
                "validation_residual_script_auc"
            ]
        ),
        "original_writer_auc": float(
            row[
                "original_writer_auc"
            ]
        ),
        "intervened_writer_auc": float(
            row[
                "intervened_writer_auc"
            ]
        ),
        "writer_auc_change": float(
            row[
                "writer_auc_change"
            ]
        ),
        "cross_macro_auc_change": float(
            row[
                "cross_macro_auc_change"
            ]
        ),
        "cross_pooled_auc_change": float(
            row[
                "cross_pooled_auc_change"
            ]
        ),
        "targeted_cross_reliance": float(
            row[
                "targeted_cross_reliance"
            ]
        ),
        "random_cross_reliance_mean": float(
            row[
                "random_cross_reliance_mean"
            ]
        ),
        "targeted_to_random_reliance_ratio": float(
            row[
                "targeted_to_random_reliance_ratio"
            ]
        ),
        "random_residual_script_auc_mean": float(
            row[
                "random_residual_script_auc_mean"
            ]
        ),
    }


notebook21_summary = {
    "experiment": (
        "Foundation Script-Reliance Audit "
        "for Cross-Script Writer Verification"
    ),
    "models": [
        "dinov2_s",
        "dinov2_b",
        "dinov2_l",
        "dinov2_g",
    ],
    "representation_of_primary_interest": (
        "PCA225 followed by writer LDA"
    ),
    "official_test_used": False,
    "fixed_rank_diagnostic": {
        "rank": 56,
        "purpose": (
            "Predefined common-rank diagnostic "
            "with dimension-matched random controls"
        ),
        "interpretation": (
            "Large rank-56 effects are not interpreted "
            "as pure script reliance because dose-response "
            "showed that script accessibility collapsed "
            "at much smaller ranks."
        ),
    },
    "confirmatory_rank_selection": {
        "protocol": (
            "Development-only writer-disjoint "
            "subspace-train and monitor split"
        ),
        "criterion": (
            "First rank whose effective monitor script "
            "AUC entered the 95 percent null-AUC band"
        ),
        "monitor_writers": int(
            len(
                monitor_writers
            )
        ),
        "subspace_train_writers": int(
            len(
                subspace_train_writers
            )
        ),
        "monitor_script_counts": (
            monitor_script_counts.tolist()
        ),
        "null_auc_upper_band": float(
            AUC_CHANCE_UPPER
        ),
    },
    "confirmatory_results": (
        confirmatory_rows
    ),
    "conclusion": {
        "supported": (
            "Script accessibility and pairwise "
            "verification-decision reliance are distinct. "
            "For DINOv2-S, B, and L PCA225-LDA "
            "representations, development-selected minimal "
            "script-associated interventions substantially "
            "reduced validation script accessibility while "
            "leaving writer-verification performance "
            "essentially unchanged."
        ),
        "not_supported": (
            "The strong writer-verification performance "
            "of PCA225-LDA foundation representations "
            "is not supported as materially dependent "
            "on the minimal linearly script-associated "
            "subspace."
        ),
        "dinov2_g_caveat": (
            "The development-selected rank for DINOv2-G "
            "did not reduce validation script accessibility "
            "to near chance, so complete script-removal "
            "reliance is inconclusive for that model."
        ),
        "rank56_caveat": (
            "Aggressive 56-dimensional removal produced "
            "larger verification changes, but script "
            "decodability had already collapsed at much "
            "smaller ranks. Those later effects therefore "
            "cannot be attributed purely to script."
        ),
        "interpretation_rule": (
            "Treat script-subspace intervention as a "
            "representation and decision-sensitivity "
            "diagnostic, not as a causal effect estimate."
        ),
    },
    "next_direction": (
        "Do not prioritize decision-stable script "
        "adaptation from these results. Evaluate "
        "foundation-verifier reliability and uncertainty "
        "against strong score-only baselines before "
        "considering a new uncertainty method."
    ),
}

summary_path = (
    NOTEBOOK21_REPORT_DIR
    / "notebook21_experiment_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        notebook21_summary,
        file,
        indent=2,
    )

saved_report_paths.append(
    summary_path
)

assert all(
    path.exists()
    for path in saved_report_paths
)

print(
    "Summary:",
    summary_path.relative_to(
        PROJECT_ROOT
    ),
)

print(
    "Total saved artifacts:",
    len(
        saved_report_paths
    ),
)

print(
    "Official test used:",
    notebook21_summary[
        "official_test_used"
    ],
)

print()

print(
    development_selected_validation_df[
        [
            "model",
            "selected_rank",
            "validation_residual_script_auc",
            "writer_auc_change",
            "cross_macro_auc_change",
            "targeted_to_random_reliance_ratio",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)

Summary: reports\foundation_script_reliance_audit\notebook21_experiment_summary.json
Total saved artifacts: 19
Official test used: False

   model  selected_rank  validation_residual_script_auc  writer_auc_change  cross_macro_auc_change  targeted_to_random_reliance_ratio
dinov2_s              3                        0.499681           0.001175                0.000306                           0.655472
dinov2_b              3                        0.573262           0.001434                0.000480                           0.576811
dinov2_l              3                        0.580038           0.001310                0.000323                           0.625742
dinov2_g              1                        0.741709           0.000791               -0.000113                           0.997275


# Final Summary — Foundation Script-Reliance Audit

## Research Question

This notebook investigated whether strong foundation-based cross-script writer verifiers merely **encode script information** or whether their pairwise verification decisions materially **depend on script-associated representation directions**.

The audit covered DINOv2-S, B, L, and G at three representation stages:

- frozen foundation embeddings,
- direct writer-LDA,
- PCA225 → writer-LDA.

The official test split remained untouched.

## Main Findings

### 1. Strong writer verification does not require low script accessibility

The strongest PCA225 → writer-LDA representations achieved high writer-verification performance while retaining almost perfectly linearly decodable script information.

Validation writer AUCs were:

- DINOv2-S: 0.899126
- DINOv2-B: 0.914989
- DINOv2-L: 0.894749
- DINOv2-G: 0.918100

At the same time, script-probe AUC was approximately 1.0 for all four representations.

Therefore, high writer-verification accuracy and low script accessibility are not equivalent properties.

### 2. Aggressive 56-D intervention was not sufficient evidence of script reliance

A predefined 56-D script-associated intervention substantially reduced script decodability and produced measurable verification changes, especially for DINOv2-S, B, and L.

However, the rank trajectory showed that script decodability had already approached chance after removing only a few directions, while the large verification degradation appeared mainly at substantially larger removal ranks.

Therefore, the large rank-56 performance drops cannot be attributed purely to script information. Later iterative directions likely overlap with broader writer/content geometry.

The rank-56 experiment is retained as an exploratory high-rank sensitivity diagnostic, not as the primary estimate of script reliance.

### 3. Development-only minimal-rank intervention separated accessibility from decision reliance

To avoid selecting an intervention rank from validation behavior, each model received a rank chosen only from a writer-disjoint development subspace-train/monitor split.

The first rank entering the 95% null-AUC band was:

- DINOv2-S: 3
- DINOv2-B: 3
- DINOv2-L: 3
- DINOv2-G: 1

For DINOv2-S, B, and L, these very small interventions strongly reduced validation script accessibility:

- S: residual script AUC = 0.499681
- B: residual script AUC = 0.573262
- L: residual script AUC = 0.580038

Yet writer verification was essentially unchanged:

- S writer AUC change = +0.001175; cross-macro change = +0.000306
- B writer AUC change = +0.001434; cross-macro change = +0.000480
- L writer AUC change = +0.001310; cross-macro change = +0.000323

Their targeted pair-score sensitivity was also smaller than dimension-matched random perturbations:

- S targeted/random reliance ratio = 0.655
- B targeted/random reliance ratio = 0.577
- L targeted/random reliance ratio = 0.626

This provides the strongest internal evidence in this notebook that **linear script accessibility and pairwise verification-decision reliance are distinct properties**.

### 4. DINOv2-G remains a qualified case

The development-selected DINOv2-G rank was 1.

Its writer-verification effect was negligible:

- writer AUC change = +0.000791
- cross-macro AUC change = -0.000113
- targeted/random reliance ratio = 0.997

However, validation script AUC remained 0.741709 after the intervention.

Therefore, the experiment does not establish complete script removal for DINOv2-G. Its minimal selected script direction appears decision-neutral, but stronger claims about complete script-accessibility removal and reliance remain inconclusive.

### 5. The intervention was not simply destroying fixed/variable-text information

Under the 56-D diagnostic, script accessibility dropped dramatically, whereas fixed/variable text-regime accessibility remained high, approximately 0.94–0.96 after intervention.

This argues against the intervention merely destroying all page-level information indiscriminately.

Nevertheless, `same_text` is only a coarse content-regime control and does not fully isolate script from exact textual content.

## Supported Conclusion

For the strongest DINOv2-S, B, and L PCA225 → writer-LDA verifiers, script identity is highly accessible in the representation, but the small development-selected subspace sufficient to suppress that accessibility has essentially no adverse effect on writer verification.

Thus:

**high nuisance decodability does not imply material pairwise decision reliance.**

The evidence supports treating representation accessibility and verification-decision sensitivity as separate quantities that should be measured independently.

## Not Supported

The experiments do not support the claim that the strong cross-script performance of the foundation-based verifiers materially depends on the minimal linearly script-associated subspace.

They also do not justify interpreting aggressive high-rank subspace removal as a pure script intervention once script decodability has already collapsed.

No causal effect of script is claimed.

## Novelty Guardrail

The iterative linear subspace-removal mechanism itself should be treated as an analysis tool rather than the methodological novelty.

The current candidate contribution is the writer-verification-specific distinction between:

**representation-level nuisance accessibility**  
and  
**pairwise verification-decision reliance**.

Any stronger novelty claim requires a dedicated literature review and confirmation on untouched or external data.

## Limitations

The QUWI validation set has already been used throughout model development and exploratory analysis, so these results are not unbiased final performance estimates.

The development-only rank selection reduces one source of post-hoc tuning, but final claims still require confirmation on untouched and preferably external cross-script data.

The coarse fixed/variable text-regime control also does not provide complete separation of script, content, acquisition, and other nuisance factors.

## Research Decision

The evidence does not currently justify prioritizing a new script-invariance or decision-stability adaptation method.

The next stage should instead evaluate **reliability and uncertainty for strong foundation verifiers**, beginning with strong score-only and existing uncertainty baselines.

A new uncertainty method should be proposed only if a reproducible complementary gap remains after those baselines.